<div style="color:black;
           display:fill;
           border-radius:5px;
           background-color:#00b3ff;
           font-size:300%;
           font-family:Verdana;   
           letter-spacing:0.5px">

<p style="font-size:30px;text-align:left">Passenger Sactisfaction Analysis and Prediction🎯</p>
</div>


<p style="text-align:center;"><img src="https://image.cnbcfm.com/api/v1/image/106918717-1627558728211-gettyimages-1233035475-778823_NA-0521-AIRTRAVEL_KKN_10920JPG.jpeg?v=1627558770&w=740&h=416" width="500" height="350">

    
The passenger is always the best the groups of client to the airline as the fare is one of the main revenue.

The survey analysis is the key to reflect the true circumstance.
    
Analyzing their feedbacks would greatly help for the airlines understand more what the passenger exactly need and
where they can improve better.
    
More importantly, predicting the passenger satisfaction also assist the CRM in airlines.
    
Therefore...

<h3>The objectives of this data analysis are:</h3>




* Perform EDA between the passenger satisfaction and all other features

* Select the best predictive models for predicting passengers satisfaction

* evaluate the high correlation factors through the great performance models


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer

import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense
import warnings

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import *
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from xgboost.sklearn import XGBClassifier
from sklearn.naive_bayes import GaussianNB

warnings.filterwarnings("ignore")

In [ ]:
df_train = pd.read_csv('../input/airline-passenger-satisfaction/train.csv')
df_test = pd.read_csv('../input/airline-passenger-satisfaction/test.csv')
df_train.drop(['Unnamed: 0','id'],axis=1,inplace=True)
df_test.drop(['Unnamed: 0','id'],axis=1,inplace=True)

print('train dataset size:',len(df_train))
print('test dataset size:',len(df_test))

In [ ]:
df_train.info()

In [ ]:
df_train.isna().sum().sort_values(ascending=False)

In [ ]:
df_test.info()

In [ ]:
df_test.isna().sum().sort_values(ascending=False)

In [ ]:
df_train.describe().T

In [ ]:
df_test.describe().T

In [ ]:
ax = sns.countplot(x="satisfaction", data=df_train)

In [ ]:
numerics = ['int64','float64']

train_con_col = df_train.select_dtypes(include = numerics).columns
train_cat_col = df_train.select_dtypes(include = "object").columns
test_con_col = df_test.select_dtypes(include = numerics).columns
test_cat_col = df_test.select_dtypes(include = "object").columns

<div style="color:black;
           display:fill;
           border-radius:5px;
           background-color:#7bade3;
           font-size:300%;
           font-family:Verdana;
           letter-spacing:0.5px">
<a class="anchor" id="1"></a> 
<p style="font-size:30px;text-align:left">Data Visualization📊</p>

</div>


In [ ]:
fig, axs = plt.subplots(9, 2, figsize=(20,50))
fig.tight_layout(pad=4.0)

for f,ax in zip(train_con_col,axs.ravel()):
    sns.set(font_scale = 2)
    ax=sns.histplot(ax=ax,data=df_train,x=df_train[f],kde=True)
    ax.set_title('Feature:'+ f)

## To see more clear in delay features, the distribution plot used:

In [ ]:
sns.distplot(df_train['Arrival Delay in Minutes'])

In [ ]:
sns.distplot(df_train['Departure Delay in Minutes'])

In [ ]:
fig, axs = plt.subplots(6, 3, figsize=(20,40))
fig.tight_layout(pad=3.0)

for f,ax in zip(train_con_col,axs.ravel()):
    sns.set(font_scale = 2)
    ax=sns.boxplot(ax=ax,data=df_train,y=df_train[f])

It can be seeing that the delay time minutes exists so many outliers.

Generally, passenger satisfy with the baggage handling and inflight service where the score ranges from 3 to 5 while others mainly stay within 2 to 4 scores.

For more further analysis, focusing on the satisfaction,

one more categorical variable added in barplot to see if the key relation appear.

In [ ]:
def detail_barplot(category):
    fig, axs = plt.subplots(9, 2, figsize=(20, 60))
    
    fig.tight_layout(pad=3.0)
    for feature,ax in zip(train_con_col,axs.ravel()):
        ax = sns.barplot(ax=ax,x="satisfaction", y=feature, hue=category,palette= 'muted', data=df_train)

detail_barplot("Gender")

Based on the result for gender,

the long flight distance make them more likely to satisfy the trip.

But in average of delay time, they may not satisfy when time is 12.5 mins above.

Between the male and female, there are no significant patterns.

In [ ]:
detail_barplot("Customer Type")

For loyal customer:
* Age around 40
* Satisfy on high flight distance,seat comfort, inflight entertainment, cleanliness

For disloyal customer:
* Age around 30
* Satisfy on Inflight wifi service, ease of online booking

Again, they generally feel neutral or dissatisfied when the delay time minutes is 12.5 or above.

In [ ]:
detail_barplot("Type of Travel")

For personal travel:

* satisfy on inflight wifi services, ease of online booking

* Average of delay time is 7.5 minutes for satisfied group

For business travel:

* satisfy on flight distance, online boarding, seat comfort, inflight entertainment, on-board service, cleanliness


In [ ]:
detail_barplot("Class")

For Business class:
Satisfy on high flight distance
General flight service score is higher than other class as it is a business class
Inflight wifi service is slight lower

For other classes, no any obvious findings found.

## To conclude..
👏Good to see no any relatively high neutral or dissatisfied scores in each category of flight experience.

Furthermore, the passengers are more likely acceptable for delay around 12.5 mins or below.

In [ ]:
for cat in train_cat_col:
    le = LabelEncoder()
    df_train[cat] = le.fit_transform(df_train[cat])
    df_test[cat] = le.fit_transform(df_test[cat])

plt.figure(figsize=(20, 8))
sns.set(font_scale = 1.25)
ax = sns.heatmap(df_train.corr().round(2),vmin=-1, vmax=1, annot=True, cmap='BrBG')

## Data cleansing and pre-processing

Using KNN for doing the data imputation...

In [ ]:
imputer = KNNImputer(n_neighbors=10, weights="uniform")

x_train = df_train.iloc[:,:-1]
x_train = imputer.fit_transform(x_train)
y_train = df_train.iloc[:,-1].to_numpy()

x_test = df_test.iloc[:,:-1]
x_test = imputer.fit_transform(x_test)
y_test = df_test.iloc[:,-1].to_numpy()

x_scaler = MinMaxScaler()
x_train = x_scaler.fit_transform(x_train)
x_test = x_scaler.fit_transform(x_test)

<div style="color:black;
           display:fill;
           border-radius:5px;
           background-color:#a8f0a8;
           font-size:300%;
           font-family:Verdana;
           letter-spacing:0.5px">
<a class="anchor" id="1"></a> 
<p style="font-size:30px;text-align:left">🛠️Model implementation</p>

</div>


In [ ]:
rf_clf = RandomForestClassifier()
lda_clf = LinearDiscriminantAnalysis()
svm_clf = SVC()
logisreg_clf = LogisticRegression()
GB_clf = GradientBoostingClassifier()
XGB_clf = XGBClassifier()
GNB_clf = GaussianNB()
    
clf_list = [rf_clf,lda_clf,svm_clf,logisreg_clf,GB_clf,XGB_clf,GNB_clf]
clf_name_list = ['random_forest','LDA','SupportVectorMachine','LogisticRegression','GradientBoosting','XGBoost','GaussianNaiveBayes']

for clf in clf_list:
    clf.fit(x_train,y_train)

In [ ]:
train_acc_list = []
test_acc_list = []

for clf,name in zip(clf_list,clf_name_list):
    
    y_pred_train = clf.predict(x_train)
    y_pred_test = clf.predict(x_test)
    
    print('***************************************************************************')
    print(name,': \n')

    print('Training part:')
    print(classification_report(y_train, y_pred_train,
                                    target_names=['neutral or dissatisfaction', 'satisfaction']))
    print('Testing part:')
    print(classification_report(y_test, y_pred_test,
                                    target_names=['neutral or dissatisfaction', 'satisfaction']))
        
    train_acc_list.append(accuracy_score(y_train, y_pred_train))
    test_acc_list.append(accuracy_score(y_test, y_pred_test))

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(10,20))
line = np.linspace(0,1)


sns.set(font_scale=1.0)
for clf, ax,name in zip(clf_list, axes.flatten(),clf_name_list):
    plot_roc_curve(clf, x_test, y_test, ax=ax)
    ax.plot(line, line, color='red', linestyle='dashed')
    ax.title.set_text(name)
fig.tight_layout(pad=1.0)
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(10,20))

sns.set(font_scale=1)

for clf, ax,name in zip(clf_list, axes.flatten(),clf_name_list):
    plot_confusion_matrix(clf, x_test, y_test, ax=ax, cmap='Purples',
                          display_labels=['neutral or dissatisfaction', 'satisfaction'])  
    
    ax.title.set_text(name)
    plt.grid(False)
fig.tight_layout(pad=1)
fig.delaxes(axes[3][1])
plt.show()

In [ ]:
plt.figure(figsize=(15,10))

n = np.arange(7)
width = 0.3

plt.bar(n, train_acc_list, color = 'green',
        width = width, edgecolor = 'black',
        label='train')
for i in range(len(train_acc_list)):
        plt.text(i,train_acc_list[i].round(2)+0.01,train_acc_list[i].round(2)*100,
                 ha = 'center',color = 'blue')

plt.bar(n + width, test_acc_list, color = 'orange',
        width = width, edgecolor = 'black',
        label='test')

for i in range(len(test_acc_list)):
        plt.text(i+0.25,test_acc_list[i].round(2)+0.01,test_acc_list[i].round(2)*100,color = 'red')

plt.xlabel("Classifiers")
plt.ylabel("% of accuracy")
plt.title("All classifiers predictive performance")
  
plt.xticks(n + width/2,clf_name_list)
plt.legend()
  
plt.show()

Great🙌🙌, the highest test accuracy can achieve 96%.

Overall, the tree-based classifier out perform than the others.

It is worth to check which factors affect the most from them.

In [ ]:
feature_importance = clf_list[0].feature_importances_
col_name = df_train.iloc[:,:-1].columns
rf_fi ={'feature_names':col_name,'feature_importance':feature_importance}

df_plt = pd.DataFrame(rf_fi)
df_plt.sort_values(by=['feature_importance'], ascending=False,inplace=True)

plt.figure(figsize=(10,8))
sns.barplot(x=df_plt['feature_importance'], y=df_plt['feature_names'])
#plt.style.use("ggplot")
plt.xlabel('Random Forest - Feature Importance')
plt.ylabel('Feature Names')
plt.show()


In [ ]:
feature_importance = clf_list[-3].feature_importances_
col_name = df_train.iloc[:,:-1].columns
rf_fi ={'feature_names':col_name,'feature_importance':feature_importance}

df_plt = pd.DataFrame(rf_fi)
df_plt.sort_values(by=['feature_importance'], ascending=False,inplace=True)

plt.figure(figsize=(10,8))
sns.barplot(x=df_plt['feature_importance'], y=df_plt['feature_names'])
plt.style.use("ggplot")
plt.xlabel('GradientBoosting - Feature Importance')
plt.ylabel('Feature Names')
plt.show()

In [ ]:
feature_importance = clf_list[-2].feature_importances_
col_name = df_train.iloc[:,:-1].columns
rf_fi ={'feature_names':col_name,'feature_importance':feature_importance}

df_plt = pd.DataFrame(rf_fi)
df_plt.sort_values(by=['feature_importance'], ascending=False,inplace=True)

plt.figure(figsize=(10,8))
sns.barplot(x=df_plt['feature_importance'], y=df_plt['feature_names'])
plt.style.use("ggplot")
plt.xlabel('XGBoost - Feature Importance')
plt.ylabel('Feature Names')
plt.show()

## DNN prediction

In [ ]:
dnn_model = Sequential()
dnn_model.add(Dense(512, input_dim=x_train.shape[1], activation='relu'))
dnn_model.add(Dense(512, activation='relu'))
dnn_model.add(Dense(512, activation='relu'))
dnn_model.add(Dense(512, activation='relu'))
dnn_model.add(Dense(1, activation='sigmoid'))

dnn_model.compile(loss='binary_crossentropy', optimizer='adam',metrics=['accuracy'])
history = dnn_model.fit(x_train, y_train, epochs=10, batch_size=256,validation_data=(x_test,y_test))

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(len(acc))

plt.plot(epochs, acc, label='training acc')
plt.plot(epochs, val_acc, label='validation acc')
plt.legend()
plt.figure()

plt.plot(epochs, loss, label='training loss')
plt.plot(epochs, val_loss, label='validation loss')
plt.legend()

## I hope you enjoy it 😃!
## Please upvote if you found that helpful 🙌🙌!